# Mechanism Tutorial 01 — Relative State X→H→X: Existence vs Expression (N=200, Γ_H)

**Conceptual model:** same 200-neuron column reused in 02 and 03.
**Question:** does a hidden relative state **exist** and when is it **expressed** in activity X?

We show:
- neutral H=H*=1 vs perturbed H_K≠1 (configured H → realized H array)
- latent H with expression disabled (Γ_H = I, b_eff = b)
- enabled Γ_H : H_K → b_eff = H_K·b → X changes
- visualization of resulting X (raster/rate)
- configured → realized → effective traceability

**Truth gates:** computational scaffold, proxy readouts, no physical-amplitude claim.
**API surface:** existing `jaxfne.emitters.simulate_edge_recurrent_izhikevich_owned_h_k_delayed` only; no new inspect API.

## Notebook grammar

setup → configured → realized → existence → latent → expressed → visualize → effective

This notebook uses package APIs through `import jaxfne as jtfne`; editable inputs are centralized; readouts are proxy-scoped; exports are JSON/PNG receipts.

## Scope: Computational Scaffold & Truth Gates

- **computational scaffold** (proxy simulation only; not calibrated to biology)
- **proxy readout** analysis (no physical amplitude claims)
- Emitter: Reduced Izhikevich (uncalibrated units)
- Source/field: proxy, `physical_amplitude_calibrated=False`

**Interpretation boundary:** hysteresis/expression demo is a scaffold dynamics study, not a biological mechanism claim.

## Colab Installation

In [ ]:
# Colab / local install: use checkout when present, otherwise pip from main.
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,opt] @ git+https://github.com/HNXJ/jaxfne.git@main"])

## Imports

In [ ]:
import os, json, hashlib
import numpy as np
import numpy as _np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxfne as jtfne
print(f"jaxfne {jtfne.__version__}")

## Runtime Constants (editable)

`TFNE_SMOKE=1` can shorten duration further.

**Same constants reused in 02/03** — change once, affect the progressive chain.

In [ ]:
SMOKE = os.environ.get('TFNE_SMOKE','0')=='1'
N_RUN = 200 if SMOKE else 200  # keep 200 for CI; docs list canonical 1000n as reference
DT_MS = 0.5
DURATION_MS = 200.0 if SMOKE else 300.0
SEED = 7

## Build Shared Conceptual Model (configured → realized)

The helper below **is identical in 02 and 03** — 01 defines the circuit once; later tutorials extend dynamics, not circuit.

In [ ]:
# Shared conceptual model — same builder reused in 01 -> 02 -> 03 (not restarted).
# Configured -> realized -> effective. Canonical 200-neuron column for speed
# (swap N=1000 for full canonical run; behaviour scales).

def make_shared_column(n=200, seed=0, dt_ms=0.5, duration_ms=300.0):
    """Build the progressive mechanism column (configured -> realized)."""
    cfg = (
        jtfne.Configuration()
        .runtime(seed=int(seed), duration_ms=float(duration_ms), dt_ms=float(dt_ms), dtype="float32")
        .areas(["V1"])
        .column("V1", layers=["L2/3", "L4", "L5", "L6"], n=int(n))
        .cell_types({"E": 0.75, "PV": 0.10, "SST": 0.08, "VIP": 0.07})
        .uniform3d(radius_mm=0.25, height_mm=1.6)
        .connectivity(within_area="all_to_all_uniform_random", within_gain=0.35, edge_seed=int(seed))
        .set_emitter("izhikevich", "cortical_eig")
        .probes(["spikes", "V_m", "source"], n_contacts=8)
        .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann", gauge="mean_zero")
    )
    return jtfne.construct(cfg)

N = 200          # canonical 1000n is the reference; 200 used for CI speed
DT_MS = 0.5
DURATION_MS = 300.0
SEED = 7
model = make_shared_column(n=N, seed=SEED, dt_ms=DT_MS, duration_ms=DURATION_MS)
print("realized:", model.summary())
print("neuron_table head:", model.neuron_table()[:2])
print("edges:", model.params["edge_list"].n_edges)
# Configured (what we declared) vs realized (what construct() built) vs effective (what simulate() produces)
import jaxfne.util as _util
try:
    print(_util.canonical_compact_summary(model))
except Exception as e:
    print("compact summary unavailable:", e)


## Declare Relative State H_K (existence)

RBS is a per-neuron relative coordinate with reference H*=1. We perturb a **single neuron** (k=0) from 1 → 2.5; all others stay neutral. Ownership mask selects which neurons carry an allocated H_K coordinate (others are fixed at reference).

In [ ]:
import jax.numpy as jnp
from jaxfne.emitters import simulate_edge_recurrent_izhikevich_owned_h_k_delayed

params = model.params["emitter"]
edges  = model.params["edge_list"]
n = params.n_neurons
n_steps = int(round(DURATION_MS / DT_MS))

h_neutral = jnp.ones(n, dtype=jnp.float32)
h_pert    = h_neutral.at[0].set(2.5)   # localized perturbation ΔH_K=1.5 on neuron 0
owner     = jnp.ones(n, dtype=jnp.float32)  # all neurons own H_K here; mask 0 would be fixed-at-1

print(f"configured H: H*=1, perturbed H_K[0]={float(h_pert[0]):.2f}, neutral rest=1.0")
print(f"realized H arrays: neutral mean {float(h_neutral.mean()):.3f}, pert mean {float(h_pert.mean()):.3f}, owner sum {int(owner.sum())}")
print(f"Gamma_H map (D1): b_eff = H_K * b  when enabled; b_eff = b when disabled (Gamma_H=I)")


## Latent H: expression disabled (Γ_H = I)

Same H values, but emitter coupling **ignores** H (`gamma_h_enabled=False` → b_eff = b). Perturbation exists in H yet is **not expressed** in X. Expect bit-identical spikes when `noise_scale=0`.

In [ ]:
key = jax.random.PRNGKey(SEED)
# noise_scale=0 makes the comparison deterministic and isolates H→X
V_lat_neutral, S_lat_neutral, src_lat_neutral, st_lat_neutral = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges, n_steps, DT_MS, key, h_k0=h_neutral, owner_mask=owner, dynamic=False, gamma_h_enabled=False, noise_scale=0.0)
V_lat_pert,    S_lat_pert,    src_lat_pert,    st_lat_pert    = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges, n_steps, DT_MS, key, h_k0=h_pert,    owner_mask=owner, dynamic=False, gamma_h_enabled=False, noise_scale=0.0)

diff_spikes_latent = int(jnp.abs(S_lat_neutral - S_lat_pert).sum())
rate_neutral_lat = float(S_lat_neutral.mean() * 1000.0 / DT_MS)
rate_pert_lat    = float(S_lat_pert.mean()    * 1000.0 / DT_MS)
print(f"latent: Δspikes (neutral vs pert, gamma OFF) = {diff_spikes_latent}  (expect 0)")
print(f"latent rates: neutral {rate_neutral_lat:.3f} Hz, pert {rate_pert_lat:.3f} Hz — identical by construction")
assert diff_spikes_latent == 0, "Gamma OFF should make H latent (no X difference with matched noise)"


## Expressed H: enable Γ_H (H_K → X)

Same H values, now `gamma_h_enabled=True` → D1 map `du = a·(H_K·b·v − u)` is active. Perturbation **is expressed** in emitter dynamics → spikes, V_m, and source Q differ.

In [ ]:
V_exp_neutral, S_exp_neutral, src_exp_neutral, st_exp_neutral = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges, n_steps, DT_MS, key, h_k0=h_neutral, owner_mask=owner, dynamic=False, gamma_h_enabled=True, noise_scale=0.0)
V_exp_pert,    S_exp_pert,    src_exp_pert,    st_exp_pert    = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges, n_steps, DT_MS, key, h_k0=h_pert,    owner_mask=owner, dynamic=False, gamma_h_enabled=True, noise_scale=0.0)

diff_spikes_expr = int(jnp.abs(S_exp_neutral - S_exp_pert).sum())
rate_neutral_expr = float(S_exp_neutral.mean() * 1000.0 / DT_MS)
rate_pert_expr    = float(S_exp_pert.mean()    * 1000.0 / DT_MS)
print(f"expressed: Δspikes (neutral vs pert, gamma ON) = {diff_spikes_expr}  (expect >0)")
print(f"expressed rates: neutral {rate_neutral_expr:.3f} Hz, pert {rate_pert_expr:.3f} Hz")
print(f"existence vs expression — existence alone (latent) left X unchanged; expression (Gamma_H) made H visible in X")
assert diff_spikes_expr > 0, "Gamma ON should make H perturbation visible in spikes"


## Visualize resulting X (proxy, uncalibrated)

Raster and rate traces show **effective** dynamics: latent = overlapping, expressed = diverging. `jaxfne.visualize` is demonstrated optionally (proxy panels, no new science).

In [ ]:
# Lightweight X visualization (raster sampling + rate) — deterministic, fast, no OOM
import matplotlib.pyplot as plt
import numpy as _np

def _plot_raster_comparison(S_a, S_b, label_a, label_b, title, time_ms):
    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    for ax, S, lab in zip(axes, [S_a, S_b], [label_a, label_b]):
        t_idx, n_idx = _np.where(_np.asarray(S) > 0)
        ax.scatter(time_ms[t_idx] if len(t_idx) else [], n_idx if len(n_idx) else [], s=2, marker='|', alpha=0.6)
        ax.set_ylabel('neuron')
        ax.set_title(lab)
        ax.grid(alpha=0.2)
    axes[-1].set_xlabel('time (ms)')
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    return fig

t = _np.arange(n_steps) * DT_MS
fig_lat = _plot_raster_comparison(S_lat_neutral, S_lat_pert, 'latent neutral (H=1, Γ off)', 'latent perturbed (H_K[0]=2.5, Γ off)', 'X→H→X latent (Γ=I): X unchanged despite H≠1', t)
plt.close(fig_lat)
fig_exp = _plot_raster_comparison(S_exp_neutral, S_exp_pert, 'expressed neutral (H=1, Γ on)', 'expressed perturbed (H_K[0]=2.5, Γ on)', 'X→H→X expressed (Γ_H active): H visible in X', t)
plt.close(fig_exp)
print("effective X: latent Δspikes=0, expressed Δspikes="+str(diff_spikes_expr))
# Show one raster explicitly
fig_exp

# Optional jaxfne.visualize bundle (proxy, may be skipped on minimal env) — uses existing API only
try:
    from jaxfne import Simulation
    # Build Signals for visualize: need a model-level run for field scaffolding
    sim_tmp = Simulation(duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED, record_sources=True, record_fields=True)
    signals_tmp = model.simulate(sim_tmp)
    bundle = jtfne.visualize(model, signals_tmp, backend="static")
    print(f"visualize bundle keys: {list(bundle.figures.keys())[:3]} ... ({len(bundle.figures)} panels, proxy)")
except Exception as e:
    print(f"visualize optional — skipped: {e}")


## Configured → Realized → Effective (traceability)

- **Configured:** what we declared (N, layers, H*=1, ΔH_K, Γ flag)
- **Realized:** what construct() + H arrays materialized (emitter params, edge_list, H vectors)
- **Effective:** what the kernel produced (spike counts, rates, sources)

No new inspect API — only `model.summary()`, `neuron_table()`, `model.params`, and returned traces.

In [ ]:
configured = {"N": N, "layers": ["L2/3","L4","L5","L6"], "H_star": 1.0, "H_pert": {"k": 0, "value": 2.5}, "Gamma_H_options": ["I (disabled)", "H_K·b (enabled)"]}
realized = {"n_units": model.summary()["n_units"], "n_edges": int(model.params["edge_list"].n_edges), "H_neutral_mean": float(h_neutral.mean()), "H_pert_mean": float(h_pert.mean()), "edge_delay_max": int(_np.max(_np.asarray(model.params["edge_list"].delay_steps)))}
effective = {"latent_delta_spikes": int(diff_spikes_latent), "expressed_delta_spikes": int(diff_spikes_expr),
             "latent_rates_hz": [round(rate_neutral_lat,3), round(rate_pert_lat,3)],
             "expressed_rates_hz": [round(rate_neutral_expr,3), round(rate_pert_expr,3)],
             "source_mean_lat_neutral": float(_np.mean(_np.asarray(src_lat_neutral))),
             "source_mean_exp_pert": float(_np.mean(_np.asarray(src_exp_pert)))}
print(json.dumps({"configured": configured, "realized": realized, "effective": effective}, indent=2))
assert realized["n_units"]==N and realized["n_edges"]>0
assert effective["latent_delta_spikes"]==0 and effective["expressed_delta_spikes"]>0
print("configured→realized→effective: verified (Δscience=0, inspect-only)")


## Export receipts & truth gates

In [ ]:
REPO_ROOT = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q/"jaxfne").is_dir() and (q/"pyproject.toml").exists()), Path.cwd())
OUTPUT_DIR = REPO_ROOT / "artifacts/tutorials/etudes/outputs/mechanism_01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {"artifact_class": "tutorial", "artifact_id": "mechanism_01_X_H_X", "tutorial": "01_relative_state_existence_vs_expression",
            "N": N, "N_canonical_reference": 1000, "H_star": 1.0, "H_pert": {"k": 0, "value": 2.5},
            "latent_delta_spikes": int(diff_spikes_latent), "expressed_delta_spikes": int(diff_spikes_expr),
            "configured": configured, "realized": realized, "effective": effective,
            "physical_amplitude_calibrated": False, "field_claim_level": "proxy_readout",
            "api_surface": "simulate_edge_recurrent_izhikevich_owned_h_k_delayed(gamma_h_enabled)", "delta_science": 0}
import jaxfne as _j; 
# json-safe
def _js(x):
    try: return _j.io.json_safe(x)
    except Exception: return json.loads(json.dumps(x, default=str))
Path(OUTPUT_DIR/"manifest.json").write_text(json.dumps(_js(manifest), indent=2))
print(f"manifest -> {OUTPUT_DIR/'manifest.json'}")
print(f"OK mechanism 01: latent {diff_spikes_latent} spikes (expect 0), expressed {diff_spikes_expr} (expect >0)")
